# **------------House Price Prediction------------**


    It covers majorly following topics: 
    - Pipelines, SimpleImputer, ColumnTransformation
    - Linear Regression, Random Forest, Gradiant Boosting, XGBoosting, LightGBM
    - Cross Validation, Hyper-Tuning
    - Residuals/Error Analysis
    - Features Importance 
    - SHAP(Ahapley Additive exPlanations)
    - Log Traget Experiments
    - Final Model Selecting and Saving

In [ ]:
#Importing required libraries/Modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#load dataset and understand
df = pd.read_csv('train.csv')

display(df.head())
display(df.info())
display(df.describe())
df.columns

In [ ]:
#Identify the target and Features variable(X,y)
# Separate features and target
X = df.drop(columns='SalePrice')
y = df["SalePrice"]
print("X shape:", X.shape)
print("y shape:", y.shape)
display(y.head())
y.describe()

In [ ]:
#Check missing values
missing_value = df.isnull().sum()
print(missing_value[missing_value >0].sort_values(ascending=False))

missing_cols = df.columns[df.isnull().any()]

# #Check numerical and categorical columns
numerical_features = df[missing_cols].select_dtypes(include=["int64", "float64"]).columns
categorical_features = df[missing_cols].select_dtypes(include=["str"]).columns
print("Numerical features:", (numerical_features))
print("Categorical features:", (categorical_features))

In [ ]:
# categorical_missing = df.select_dtypes(include="str").isnull().sum()
# categorical_missing[categorical_missing > 0].sort_values(ascending=False)
    

none_cols = [
    "Alley",
    "MasVnrType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature"
]

df[none_cols] = df[none_cols].fillna("None")

df["Electrical"] = df["Electrical"].fillna(
    df["Electrical"].mode()[0]
)

# for rechecking every unique value with frequence of column which have null value 
for col in df.columns[df.isnull().any()]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
# fill missing values in categorical columns 
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
neighborhood_median = df.groupby("Neighborhood")["LotFrontage"].transform("median")  #It gives each row the median of its own neighborhood.
df["LotFrontage"] = df["LotFrontage"].fillna(neighborhood_median)
# df.isnull().sum()[df.isnull().sum() > 0]

### ***EDA Exploratory Data Analysis***

In [ ]:
#Understand the Target: SalePrice
#Before looking at 80 features, let's understand what we're trying to predict.
df["SalePrice"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.set_theme()
sns.histplot(
    df["SalePrice"],
    kde=True
)

plt.title("Distribution of House Prices")
plt.xlabel("Sale Price")
plt.ylabel("Number of Houses")

plt.show()

In [ ]:
df["SalePrice"].skew()

In [ ]:
df["SalePrice"].sort_values(ascending=False).head(10)

In [ ]:
#Find the strongest relationships with SalePrice
correlation = df.select_dtypes(
    include=["int64", "float64"]
).corr()["SalePrice"].sort_values(ascending=False)

correlation.head(11)

In [ ]:
top_features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF"
]

for feature in top_features:
    plt.figure(figsize=(7, 5))

    sns.scatterplot(
        data=df,
        x=feature,
        y="SalePrice"
    )

    plt.title(f"{feature} vs SalePrice")
    plt.xlabel(feature)
    plt.ylabel("SalePrice")

    plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice"
)

plt.title("GrLivArea vs SalePrice")
plt.xlabel("Living Area")
plt.ylabel("Sale Price")

plt.show()

In [ ]:
df.nlargest(10, "GrLivArea")[
    ["GrLivArea", "OverallQual", "Neighborhood", "SalePrice"]
]

In [ ]:
df.loc[[1298, 523]].T

In [ ]:
df = df.drop(index=[1298, 523])


In [ ]:
df.shape

In [ ]:
#optional
#Analyze Categorical Features
neighborhood_price = (
    df.groupby("Neighborhood")["SalePrice"]
      .mean()
      .sort_values(ascending=False)
)

neighborhood_price

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=neighborhood_price.index,
    y=neighborhood_price.values
)

plt.xticks(rotation=45)
plt.title("Average Sale Price by Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("Average Sale Price")

plt.show()

In [ ]:
quality_price = (
    df.groupby("OverallQual")["SalePrice"]
      .mean()
      .sort_index()
)

quality_price

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=quality_price.index,
    y=quality_price.values
)

plt.title("Average Sale Price by Overall Quality")
plt.xlabel("Overall Quality")
plt.ylabel("Average Sale Price")

plt.show()

In [ ]:
#optional
#10 most skewed numerical features.
numerical_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

skewness = (df[numerical_cols].skew().sort_values(ascending=False))

skewness.head(15)
#high skewness here is actually telling us something about the feature distribution, not necessarily that the data is bad.

### Investigate important numerical features that's important for linear regression 
We should examine our top correlated features in groups.

Group A — House size
- GrLivArea
- TotalBsmtSF
- 1stFlrSF

These describe different aspects of house size.

Group B — Quality / capacity
- OverallQual
- GarageCars
- FullBath
- TotRmsAbvGrd

Group C — Age
- YearBuilt
- YearRemodAdd

This tells us something useful:

Price is strongly associated with quality + size + capacity + age.

That's already a meaningful business interpretation.

    GarageCars ↔ GarageArea     strongly related 
    TotalBsmtSF ↔ 1stFlrSF      strongly related 
    GrLivArea ↔ TotRmsAbvGrd    strongly related
    YearBuilt ↔ YearRemodAdd     Moderately related
These may contain overlapping information. That known after see heatmap 

For Linear Regression, highly correlated predictors can cause multicollinearity.

In [ ]:
important_features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd"
]

corr_matrix = df[important_features].corr()

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f"
)

plt.title("Correlation Between Important Numerical Features")

plt.show()

In [ ]:
categorical_cols = df.select_dtypes(include="str").columns

category_summary = []

for col in categorical_cols:
    if df[col].nunique() <= 30:
        grouped = df.groupby(col)["SalePrice"].agg(
            ["mean", "median", "count"]
        )

        category_summary.append(
            (col, grouped["mean"].max() - grouped["mean"].min())
        )

sorted(category_summary, key=lambda x: x[1], reverse=True)[:10]

### ***Train/Test split***
### ***Entering into actull ML pipeline***

In [ ]:
X = df.drop('SalePrice',axis=1)
y = df['SalePrice']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test  = train_test_split(X,y, test_size=0.2, random_state=42)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

****Identify Feature Types****

Before encoding/scaling anything, we need to separate:

    Numerical features → scaling / transformations
    Categorical features → encoding

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["str"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical:")
print(numeric_features)
print("\nCategorical:")
print(categorical_features)

### ***Build a Baseline Model Linear Regression*** 

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_tranformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num',numeric_tranformer,numerical_features),
    ('cat',categorical_transformer,categorical_features)
])

linear_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

y_pred = linear_pipeline.predict(X_test)

#Evaluate Model 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error 

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test,y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print("Linear Regression Baseline")
print("--------------------------")
print(f"MAE  : {mae:,.2f}")
print(f"MSE : {mse:,.2f}")
print(f"RMSE : {rmse:,.2f}")    
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape * 100:.2f}%")

#Lower MAE/RMSE = better. Higher R² = better.



#### What this tells us

Our Linear Regression model is already explaining about **74.4%** of the variation in house prices.

But notice:

```text
MAE  = $27.6K
RMSE = $37.6K
```

RMSE is considerably larger than MAE, which suggests that **some houses have relatively large prediction errors**.

That's useful information—we'll investigate those errors later.

### ***Ensemble: Random Forest***

In [ ]:
from sklearn.ensemble import RandomForestRegressor

numeric_tranformer = Pipeline([
    ('impute',SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_tree = ColumnTransformer([
    ('num',numeric_tranformer,numerical_features),
    ('cat',categorical_transformer,categorical_features)
])

rf_pipeline = Pipeline([
    ('preprocessor',preprocessor_tree),
    ('model',RandomForestRegressor(
        n_estimators=300,                       #300 decision trees
        random_state=42,
        n_jobs=-1                               #tells sklearn to use all available CPU cores.
        ))      
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)

#Evaluate

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest")
print("--------------------------")
print(f"MAE  : {rf_mae:,.2f}")
print(f"MSE  : {rf_mse:,.2f}")
print(f"RMSE : {rf_rmse:,.2f}")
print(f"R²   : {rf_r2:.4f}")

#### Improvement

Random Forest improved:

* **MAE:** ~15.6% lower
* **RMSE:** ~14.8% lower
* **R²:** 0.7443 → **0.8142**

### ***Gradient Boosting Regressor***

In [ ]:
# Built Gradiant Boosting Pipeline

from sklearn.ensemble import GradientBoostingRegressor

gbr_pipeline = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', GradientBoostingRegressor(
        n_estimators=300,                       #300 decision trees
        learning_rate=0.05,                     #How strongly each new tree contributes.
        max_depth=3,                            #Controls tree complexity.
        random_state=42
    ))
])

#Train + Evaluate
gbr_pipeline.fit(X_train, y_train)

gbr_pred = gbr_pipeline.predict(X_test)

gbr_mae = mean_absolute_error(y_test, gbr_pred)
gbr_mse = mean_squared_error(y_test, gbr_pred)
gbr_rmse = np.sqrt(gbr_mse)
gbr_r2 = r2_score(y_test, gbr_pred)

print("Gradient Boosting")
print("--------------------------")
print(f"MAE  : {gbr_mae:,.2f}")
print(f"MSE  : {gbr_mse:,.2f}")
print(f"RMSE : {gbr_rmse:,.2f}")
print(f"R²   : {gbr_r2:.4f}")

#### What we learned

Gradient Boosting improved:

* **MAE:** $23,259 → $23,140
* **RMSE:** $32,034 → $30,840
* **R²:** 0.8142 → *0.8278*

The MAE improvement is small, but RMSE improved substantially, meaning Gradient Boosting is doing a better job reducing some of the larger errors.

> We're using reasonable starting values, not claiming they're optimal. We'll tune them later.

### ***XGBoost Baseline***

In [ ]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline([
    ('preprocessor',preprocessor_tree),
    ('model',XGBRegressor(
        n_estimators=1000,              #number of boosting tree
        learning_rate=0.03,             #contribution of each tree
        max_depth=3,                    #Controll tree complexity
        subsample=0.8,                  #uses 80% rows per boosting round
        colsample_bytree=0.8,           #uses 80% features per tree
        objective='reg:squarederror',   #regression objective
        random_state=42,    
        n_jobs=-1                       # use all CPU cores
    ))
])

#We're intentionally using a small learning rate + many trees, which is a common boosting strategy.

xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)

#Evaluate
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost")
print("--------------------------")
print(f"MAE  : {xgb_mae:,.2f}")
print(f"MSE  : {xgb_mse:,.2f}")
print(f"RMSE : {xgb_rmse:,.2f}")
print(f"R²   : {xgb_r2:.4f}")


#### XGBoost result

| Model             |       MAE ↓ |      RMSE ↓ |          R² ↑ |
| ----------------- | ----------: | ----------: | ------------: |
| Linear Regression |     $27,552 |     $37,579 |        0.7443 |
| Random Forest     |     $23,259 |     $32,034 |        0.8142 |
| Gradient Boosting |     $23,140 |     $30,840 |        0.8278 |
| **XGBoost**       | **$22,691** | **$30,224** | **0.8346** 🏆 |


### ***Cross Validation***

In [ ]:
#Fold CV for XGBoost
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=kfold,
    scoring="r2",
    n_jobs=-1
)

print("XGBoost Cross-Validation R²")
print("--------------------------")
print("Fold scores:", cv_scores)
print("Mean R²    :", cv_scores.mean())
print("Std R²     :", cv_scores.std())

### ***XGBoost Hyperparameter Tuning***

In [ ]:
# Now we optimize the hyperparameters of the XGBoost model using RandomizedSearchCV

from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "model__n_estimators": [300, 500, 700, 1000, 1500],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}
#5 × 5 × 5 × 4 × 4 × 4 = 8000 total possible combinations

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=30,                                  # We're testing 30 randomly selected parameter combinations
    scoring="r2",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
# 30 × 5 = 150 model fits
random_search.fit(X_train, y_train)

print("Best CV R²:", random_search.best_score_)
print("\nBest Parameters:")
print(random_search.best_params_) 

#Evaluate the Tuned XGBoost
best_xgb = random_search.best_estimator_    #Give me the complete model/pipeline with best parameter combination found during the search.

tuned_pred = best_xgb.predict(X_test)      

tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_mse = mean_squared_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, tuned_pred)

print("Tuned XGBoost")
print("--------------------------")
print(f"MAE  : {tuned_mae:,.2f}")
print(f"MSE  : {tuned_mse:,.2f}")
print(f"RMSE : {tuned_rmse:,.2f}")
print(f"R²   : {tuned_r2:.4f}")

### ***LightGBM***

In [ ]:
from lightgbm import LGBMRegressor

# Create a pipeline for LightGBM
lgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", LGBMRegressor(
        n_estimators=1000,          #Up to 1000 boosting trees
        learning_rate=0.03,         #Each tree makes a small correction
        max_depth=-1,               #Controls tree complexity
        num_leaves=31,              #No explicit depth limit
        subsample=0.8,              #Uses 80% of samples per iteration
        colsample_bytree=0.8,       #Uses 80% of features
        random_state=42,
        n_jobs=-1,                  #Uses all CPU cores
        verbosity=-1
    ))
])

lgbm_pipeline.fit(X_train, y_train)

#Evaluate
lgbm_pred = lgbm_pipeline.predict(X_test)

lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
lgbm_mse = mean_squared_error(y_test, lgbm_pred)
lgbm_rmse = np.sqrt(lgbm_mse)
lgbm_r2 = r2_score(y_test, lgbm_pred)

print("LightGBM")
print("-" * 26)
print(f"MAE  : {lgbm_mae:,.2f}")
print(f"MSE  : {lgbm_mse:,.2f}")
print(f"RMSE : {lgbm_rmse:,.2f}")
print(f"R²   : {lgbm_r2:.4f}")

Good. **LightGBM is weaker than your tuned XGBoost on this dataset**, so we don't need to force it as the final model.

#### Current leaderboard

| Model             |         R² |       RMSE |        MAE |
| ----------------- | ---------: | ---------: | ---------: |
| Linear Regression |     0.7443 |     37,579 |     27,552 |
| Random Forest     |     0.8142 |     32,034 |     23,259 |
| Gradient Boosting |     0.8278 |     30,840 |     23,140 |
| **Tuned XGBoost** | **0.8281** | **30,818** | **23,117** |
| XGBoost baseline  |     0.8346 |     30,224 |     22,691 |
| LightGBM          |     0.8021 |     33,063 |     23,266 |

The baseline XGBoost test score is higher, but **don't select it based only on this test score**. The tuned model was selected using CV, where it achieved **CV R² = 0.8080**, versus the untuned model's **0.8067**.

#### Important conclusion

Your current **best validated model is Tuned XGBoost**:

```text
CV R²   = 0.8080
Test R² = 0.8281
MAE     = ₹23,117
RMSE    = ₹30,818
```

### ***Residual / Error Analysis***

In [ ]:
# Finds where is our XGBoost model making mistakes.

residuals = y_test - tuned_pred
#Positive residual → model underpredicted
#Negative residual → model overpredicted

print("Mean Residual :", residuals.mean())
print("Median Residual:", residuals.median())
print("Min Residual  :", residuals.min())
print("Max Residual  :", residuals.max())

error_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": tuned_pred,
    "Residual": residuals.values,
    "Absolute_Error": np.abs(residuals.values)
})

error_df = error_df.sort_values(
    "Absolute_Error",
    ascending=False
)

error_df.head(10)
#This will show us the 10 houses where our model performed worst.

```text
Mean Residual   = -2,192
Median Residual = -3,461

Both are negative, so your model has a slight tendency to overpredict.
But ₹2,192 is relatively small compared with house prices in this dataset, so this isn't a major problem.
```

Your largest errors are:

|  Actual | Predicted |    Error | Behavior     |
| ------: | --------: | -------: | ------------ |
| 359,100 |   243,771 | +115,329 | Underpredict |
| 190,000 |   301,935 | -111,935 | Overpredict  |
| 311,500 |   218,052 |  +93,448 | Underpredict |
| 243,000 |   150,042 |  +92,958 | Underpredict |
| 438,780 |   523,498 |  -84,718 | Overpredict  |
|  40,000 |   123,575 |  -83,575 | Overpredict  |

The interesting one is:

```text
Actual = 40,000
Predicted = 123,575
```

That's a **huge relative error**.

This is exactly why looking only at R² isn't enough.

In [ ]:
# We need a residual plot now

sns.set_theme()
plt.figure(figsize=(8, 5))

plt.scatter(tuned_pred, residuals, alpha=0.6)

plt.axhline(y=0, linestyle="--")

plt.xlabel("Predicted SalePrice")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted SalePrice")

plt.show()

***We want the points to be randomly scattered around 0.***

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(y_test, tuned_pred, alpha=0.6)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted SalePrice")

plt.show()

***The closer the points are to that line, the better.***

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, tuned_pred)

print(f"MAPE: {mape * 100:.2f}%")

### ***Feature Importance & Model Interpretation***
```text
We'll use the tuned XGBoost model (best_xgb).
```

In [ ]:
# 1. Extract feature importance

# Get fitted preprocessor
preprocessor_fitted = best_xgb.named_steps["preprocessor"]

# Get fitted XGBoost model
xgb_model = best_xgb.named_steps["model"]

# Get feature names after preprocessing
feature_names = preprocessor_fitted.get_feature_names_out()

# Get importance scores
importance = xgb_model.feature_importances_

# Create DataFrame
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

# Sort
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

feature_importance_df.head(20)

In [ ]:
top20 = feature_importance_df.head(20).sort_values(
    "Importance"
)

plt.figure(figsize=(10, 8))

plt.barh(
    top20["Feature"],
    top20["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 20 XGBoost Feature Importances")

plt.show()

In [ ]:
# Merging columns who splits during one hot encoding 


# importance_df = feature_importance_df.copy()

# importance_df["Original_Feature"] = (
#     importance_df["Feature"]
#     .str.replace(r"^(num__|cat__)", "", regex=True)
#     .str.split("_")
#     .str[0]
# )

# grouped_importance = (
#     importance_df
#     .groupby("Original_Feature")["Importance"]
#     .sum()
#     .sort_values(ascending=False)
# )

# grouped_importance.head(20)


# More reliable 
numeric_features_original = numeric_features
categorical_features_original = categorical_features

importance_df = feature_importance_df.copy()

importance_df["Original_Feature"] = importance_df["Feature"].apply(
    lambda x: next(
        (
            feature
            for feature in categorical_features_original
            if x.startswith(f"cat__{feature}_")
        ),
        next(
            (
                feature
                for feature in numeric_features_original
                if x == f"num__{feature}"
            ),
            x
        )
    )
)

grouped_importance = (
    importance_df
    .groupby("Original_Feature")["Importance"]
    .sum()
    .sort_values(ascending=False)
)

print(grouped_importance.head(20))

### ***SHAP (SHapley Additive exPlanations).***
Shap can tell us:

This particular house:

```text
BsmtQual_Ex      → +₹42,000
KitchenQual_Ex   → +₹25,000
Neighborhood_X   → +₹18,000
GrLivArea        → +₹31,000
...
                  ↓
              Prediction
```

In [ ]:
import shap

xgb_model = best_xgb.named_steps["model"]       #get xgb model from best_xgb

X_test_transformed = best_xgb.named_steps["preprocessor"].transform(X_test)     #this tranform X_test data into maching readable form with the help of preprocessor

feature_names = best_xgb.named_steps["preprocessor"].get_feature_names_out()    #Get feature names

explainer = shap.TreeExplainer(xgb_model)       #Create SHAP explainer

shap_values = explainer.shap_values(X_test_transformed)       #SHAP calculates a contribution value for every feature for every house.

print("SHAP ready!")

#### The complete flow

```text
X_test
  │
  ▼
best_xgb
  │
  ├── Preprocessor
  │      ├── Numerical → Imputer
  │      └── Categorical → Imputer → OneHotEncoder
  │
  ▼
X_test_transformed
  │
  ▼
XGBoost Model
  │
  ▼
SHAP TreeExplainer
  │
  ▼
SHAP Values
  │
  ├── Which features matter?
  ├── How much do they matter?
  ├── Do they increase price?
  └── Do they decrease price?
```

#### The key difference

Your previous feature importance told us:

> **"Which features does the model generally consider important?"**

SHAP can tell us:

> **"Why did the model predict this particular house at this particular price?"**



In [ ]:
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    max_display=20
)

***Especially: These features make positive effects***

```text
ExterQual
BsmtQual
KitchenQual
```

### ***We're doing 5-fold cross-validation only for the best‑performing models***

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Gradient Boosting": gbr_pipeline,
    "XGBoost Baseline": xgb_pipeline,
    "Tuned XGBoost": best_xgb
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="r2",
        n_jobs=-1
    )

    print(f"\n{name}")
    print("Fold R²:", scores)
    print("Mean R²:", scores.mean())
    print("Std R²:", scores.std())

Excellent. Now we have enough evidence to make the **model-selection decision**.

#### 5-Fold CV results

| Model                | Mean CV R² |        Std |    Test R² |
| -------------------- | ---------: | ---------: | ---------: |
| Gradient Boosting    |     0.7785 |     0.0462 |     0.8278 |
| Tuned XGBoost        |     0.8007 | **0.0397** |     0.8281 |
| **XGBoost Baseline** | **0.8067** |     0.0440 | **0.8346** |

#### 🏆 Winner: XGBoost Baseline

Why?

* **Best CV R²:** 0.8067
* **Best test R²:** 0.8346
* **Best test RMSE:** 30,224
* **Best test MAE:** 22,691

The tuned version has slightly lower CV variance (`0.0397` vs `0.0440`), but its average CV performance and test performance are both worse.

So we should **not force the tuned model to be the final model just because it was tuned**.

#### Our final candidate

```python
final_model = xgb_pipeline
```


This is a good real-world lesson:

> **Hyperparameter tuning does not guarantee better generalization.**


### ***Now we'll run the log-target experiment.***

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#Transform the target
y_train_log = np.log1p(y_train)

print(y_train_log.head())
print("Original skew:", y_train.skew())
print("Log skew:", y_train_log.skew())

#Train XGBoost on log target
xgb_log = clone(xgb_pipeline)
xgb_log.fit(X_train, y_train_log)

# Predict in log scale
y_pred_log = xgb_log.predict(X_test)

# Convert predictions back to original price scale
y_pred_log = np.expm1(y_pred_log)

# Evaluate on original price scale
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

print("Log-Target XGBoost")
print("------------------")
print(f"MAE  : {mae_log:,.2f}")
print(f"RMSE : {rmse_log:,.2f}")
print(f"R²   : {r2_log:.4f}")


Exactly — and your **log-target XGBoost is clearly better** on this test split.

#### Your final comparison

| Model                  |      MAE ↓ |     RMSE ↓ |        R² ↑ |
| ---------------------- | ---------: | ---------: | ----------: |
| Raw-target XGBoost     |     22,691 |     30,224 |      0.8346 |
| **Log-target XGBoost** | **21,293** | **28,813** |  **0.8497** |
| Improvement            |  **~6.2%** |  **~4.7%** | **+0.0151** |

So for now, **log-target XGBoost is our winner.** 🏆




### ***Validate the log-target XGBoost with 5-fold cross-validation.***

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone
import numpy as np

# Fresh log-target XGBoost
xgb_log_cv = clone(xgb_pipeline)

# 5-fold shuffled CV
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Cross-validation on log target
cv_scores_log = cross_val_score(
    xgb_log_cv,
    X_train,
    y_train_log,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

print("Log-Target XGBoost CV R² scores:")
print(cv_scores_log)

print("\nMean CV R²:", cv_scores_log.mean())
print("Std CV R² :", cv_scores_log.std())

Excellent. This is a **very good result**. Your log-target XGBoost is not only better on the test set, but also more stable across folds.

#### Comparison

| Model                  |  Test R² ↑ | CV Mean R² ↑ |   CV Std ↓ |
| ---------------------- | ---------: | -----------: | ---------: |
| Raw XGBoost            |     0.8346 |       0.8067 |     0.0440 |
| **Log-target XGBoost** | **0.8497** |   **0.8172** | **0.0226** |

#### What this tells us

**1. Better generalization**

```text
0.8067 → 0.8172
```

CV R² improved by about **0.0105**.

**2. Much more stable**

```text
0.0440 → 0.0226
```

The CV standard deviation was almost cut in half. That's a particularly good sign.

**3. Test performance also improved**

```text
R²   : 0.8346 → 0.8497
RMSE : 30,224 → 28,813
MAE  : 22,691 → 21,293
```

So we now have consistent evidence that the **log-target approach is better**.

### ***Just test log-target + tuned XGBoost.***

In [ ]:
from sklearn.base import clone
import numpy as np

# Clone the tuned XGBoost pipeline
xgb_log_tuned = clone(best_xgb)

# Train on log-transformed target
xgb_log_tuned.fit(X_train, y_train_log)

# Predict
y_pred_log_tuned = xgb_log_tuned.predict(X_test)

# Convert back to original price
y_pred_log_tuned = np.expm1(y_pred_log_tuned)

# Evaluate
mae_log_tuned = mean_absolute_error(y_test, y_pred_log_tuned)
rmse_log_tuned = np.sqrt(
    mean_squared_error(y_test, y_pred_log_tuned)
)
r2_log_tuned = r2_score(y_test, y_pred_log_tuned)

print("Tuned Log-Target XGBoost")
print("------------------------")
print(f"MAE  : {mae_log_tuned:,.2f}")
print(f"RMSE : {rmse_log_tuned:,.2f}")
print(f"R²   : {r2_log_tuned:.4f}")

```text
Perfect. This confirms something important:Now tuning did not improve our log-target model.
🏆 Winner: Log-target XGBoost
```

### ***Final Validation***

Before we lock it, let's check whether the model has any systematic prediction bias.

We'll examine:

- Actual vs Predicted
- Residual distribution
- Residual vs Predicted

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

residuals = y_test - y_pred_log

# 1. Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_log, alpha=0.6)

min_val = min(y_test.min(), y_pred_log.min())
max_val = max(y_test.max(), y_pred_log.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.title("Actual vs Predicted — Log XGBoost")
plt.show()


# 2. Residual distribution
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30)

plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution — Log XGBoost")
plt.show()


# 3. Residual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_pred_log, residuals, alpha=0.6)

plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Sale Price")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted — Log XGBoost")
plt.show()


print("Mean residual  :", residuals.mean())
print("Median residual:", np.median(residuals))
print("Min residual   :", residuals.min())
print("Max residual   :", residuals.max())

### ***Final Model Training*** 

In [ ]:
# Prepare the complete cleaned data set

# Craete the final log target
y_log = np.log1p(y)

# print("Original skew:", y.skew())
# print("Log skew:", y_log.skew())


# Create the fresh final model
from sklearn.base import clone

final_model = clone(xgb_pipeline) 


# Train all cleaned data
final_model.fit(X,y_log)


# Make a test prediction just to verify it works
final_test_pred_log = final_model.predict(X_test)
final_test_pred = np.expm1(final_test_pred_log)

print(final_test_pred[:10])

### ***Saving the model***

In [ ]:
# This is important for deployment

import joblib

joblib.dump(final_model, "house_price_xgb_model.pkl")

print("Model saved successfully!")